In [29]:
import pandas as pd

from pathlib import Path


# Diretório principal do projeto
PROJECT_ROOT = Path.cwd().parent


# Origem: arquivos JSON da camada Bronze
SOURCE_DIR = (
    PROJECT_ROOT
    / "data"
    / "bronze"
    / "open_meteo"
    / "forecast"
)


# Destino: arquivos Parquet da camada Silver
TARGET_DIR = (
    PROJECT_ROOT
    / "data"
    / "silver"
    / "open_meteo"
    / "forecast"
)


# Conferir os caminhos
print("Diretório do projeto:", PROJECT_ROOT)
print("Origem Bronze:", SOURCE_DIR)
print("Destino Silver:", TARGET_DIR)

print("\nBronze existe?", SOURCE_DIR.exists())
print("Silver existe?", TARGET_DIR.exists())

Diretório do projeto: /home/mariarita-resende/Projetos/open_meteo_ingestion
Origem Bronze: /home/mariarita-resende/Projetos/open_meteo_ingestion/data/bronze/open_meteo/forecast
Destino Silver: /home/mariarita-resende/Projetos/open_meteo_ingestion/data/silver/open_meteo/forecast

Bronze existe? True
Silver existe? True


In [30]:
def get_forecast_files(directory: Path) -> list[Path]:
    if not directory.exists():
        raise FileNotFoundError(
            f"Diretório Bronze não encontrado: {directory}"
        )

    return sorted(directory.rglob("*.json"))


forecast_files = get_forecast_files(SOURCE_DIR)

print(f"Total de arquivos JSON encontrados: {len(forecast_files)}")

for path in forecast_files[:5]:
    print(path)

Total de arquivos JSON encontrados: 74
/home/mariarita-resende/Projetos/open_meteo_ingestion/data/bronze/open_meteo/forecast/20260423/amsterdam_20260423.json
/home/mariarita-resende/Projetos/open_meteo_ingestion/data/bronze/open_meteo/forecast/20260423/barcelona_20260423.json
/home/mariarita-resende/Projetos/open_meteo_ingestion/data/bronze/open_meteo/forecast/20260423/berlin_20260423.json
/home/mariarita-resende/Projetos/open_meteo_ingestion/data/bronze/open_meteo/forecast/20260423/brussels_20260423.json
/home/mariarita-resende/Projetos/open_meteo_ingestion/data/bronze/open_meteo/forecast/20260423/budapest_20260423.json


In [31]:
def build_forecast_dataframe(path: Path) -> pd.DataFrame:
    path = Path(path)

    raw = pd.read_json(path)
    hourly = raw.get("hourly", {})

    if isinstance(hourly, pd.Series):
        hourly = hourly.to_dict()

    if not isinstance(hourly, dict):
        return pd.DataFrame()

    times = pd.Series(hourly.get("time", [])).tolist()

    if not times:
        return pd.DataFrame()

    field_mapping = {
        "temperature_2m": "temperature_celsius",
        "apparent_temperature": "feels_like_celsius",
        "relative_humidity_2m": "relative_humidity_pct",
        "precipitation_probability": "precipitation_probability",
        "precipitation": "precipitation_mm",
        "weather_code": "weather_code",
        "wind_speed_10m": "wind_speed_10m",
        "wind_direction_10m": "wind_direction_10m",
    }

    data = {
        "source_name": ["open_meteo"] * len(times),
        "source_type": ["forecast"] * len(times),
        "city_name": [path.stem.rsplit("_", 1)[0]] * len(times),
        "ingestion_date": [path.parent.name] * len(times),
        "observation_time": [f"{time}Z" for time in times],
    }

    for source_field, target_field in field_mapping.items():
        values = hourly.get(source_field, [])

        if not isinstance(values, list):
            values = []

        values = values[:len(times)]

        if len(values) < len(times):
            values.extend([None] * (len(times) - len(values)))

        data[target_field] = values

    return pd.DataFrame(data)

In [32]:
sample_path = forecast_files[0]

sample_df = build_forecast_dataframe(sample_path)

print("Arquivo testado:", sample_path.name)
print("Quantidade de linhas:", len(sample_df))
print("Quantidade de colunas:", len(sample_df.columns))
print("\nColunas do DataFrame:")

print(sample_df.columns.tolist())

print("\nPrimeiras 5 linhas:")

display(sample_df.head())

Arquivo testado: amsterdam_20260423.json
Quantidade de linhas: 168
Quantidade de colunas: 13

Colunas do DataFrame:
['source_name', 'source_type', 'city_name', 'ingestion_date', 'observation_time', 'temperature_celsius', 'feels_like_celsius', 'relative_humidity_pct', 'precipitation_probability', 'precipitation_mm', 'weather_code', 'wind_speed_10m', 'wind_direction_10m']

Primeiras 5 linhas:


,source_name,source_type,city_name,ingestion_date,observation_time,temperature_celsius,feels_like_celsius,relative_humidity_pct,precipitation_probability,precipitation_mm,weather_code,wind_speed_10m,wind_direction_10m
0,open_meteo,forecast,amsterdam,20260423,2026-04-23T00:00Z,8.8,6.1,80,0,0.0,0,10.8,20
1,open_meteo,forecast,amsterdam,20260423,2026-04-23T01:00Z,8.7,6.1,78,0,0.0,0,9.4,36
2,open_meteo,forecast,amsterdam,20260423,2026-04-23T02:00Z,8.4,5.6,81,0,0.0,0,11.2,33
3,open_meteo,forecast,amsterdam,20260423,2026-04-23T03:00Z,8.0,5.3,82,0,0.0,0,10.4,28
4,open_meteo,forecast,amsterdam,20260423,2026-04-23T04:00Z,8.0,5.4,82,0,0.0,0,9.4,14


In [33]:
print("Valores ausentes por coluna:")
print(sample_df.isna().sum())

print("\nTotal de valores ausentes:")
print(sample_df.isna().sum().sum())

print("\nLinhas com pelo menos um valor ausente:")
print(sample_df.isna().any(axis=1).sum())

Valores ausentes por coluna:
source_name                  0
source_type                  0
city_name                    0
ingestion_date               0
observation_time             0
temperature_celsius          0
feels_like_celsius           0
relative_humidity_pct        0
precipitation_probability    0
precipitation_mm             0
weather_code                 0
wind_speed_10m               0
wind_direction_10m           0
dtype: int64

Total de valores ausentes:
0

Linhas com pelo menos um valor ausente:
0


In [34]:
TARGET_DIR.mkdir(parents=True, exist_ok=True)

total_rows = 0
files_written = 0
files_skipped = 0
errors = []

for path in forecast_files:
    try:
        df = build_forecast_dataframe(path)

        if df.empty:
            files_skipped += 1
            print(f"Arquivo ignorado (sem dados): {path.name}")
            continue

        output_dir = TARGET_DIR / path.parent.name
        output_dir.mkdir(parents=True, exist_ok=True)

        output_path = output_dir / f"{path.stem}.parquet"

        df.to_parquet(
            output_path,
            engine="pyarrow",
            index=False
        )

        total_rows += len(df)
        files_written += 1

    except Exception as error:
        errors.append((str(path), str(error)))
        print(f"Erro ao processar {path.name}: {error}")

print("\nProcessamento concluído!")
print(f"Arquivos JSON encontrados: {len(forecast_files)}")
print(f"Arquivos Parquet gravados: {files_written}")
print(f"Arquivos ignorados: {files_skipped}")
print(f"Total de linhas gravadas: {total_rows}")
print(f"Erros encontrados: {len(errors)}")
print(f"Diretório Silver: {TARGET_DIR}")


Processamento concluído!
Arquivos JSON encontrados: 74
Arquivos Parquet gravados: 74
Arquivos ignorados: 0
Total de linhas gravadas: 12432
Erros encontrados: 0
Diretório Silver: /home/mariarita-resende/Projetos/open_meteo_ingestion/data/silver/open_meteo/forecast


In [35]:
parquet_files = sorted(TARGET_DIR.rglob("*.parquet"))

print(f"Total de arquivos Parquet encontrados: {len(parquet_files)}")

assert len(parquet_files) == len(forecast_files), (
    "A quantidade de arquivos Parquet não corresponde "
    "à quantidade de arquivos JSON."
)

print("\nArquivos encontrados:")
for path in parquet_files[:5]:
    print(path.name)

# Ler um arquivo Parquet para validar
sample_parquet = pd.read_parquet(parquet_files[0], engine="pyarrow")

print("\nArquivo validado:", parquet_files[0].name)
print("Quantidade de linhas:", len(sample_parquet))
print("Quantidade de colunas:", len(sample_parquet.columns))

print("\nColunas:")
print(sample_parquet.columns.tolist())

print("\nPrimeiras 5 linhas:")
display(sample_parquet.head())

print("\nValores ausentes:")
print(sample_parquet.isna().sum())

Total de arquivos Parquet encontrados: 74

Arquivos encontrados:
amsterdam_20260423.parquet
barcelona_20260423.parquet
berlin_20260423.parquet
brussels_20260423.parquet
budapest_20260423.parquet

Arquivo validado: amsterdam_20260423.parquet
Quantidade de linhas: 168
Quantidade de colunas: 13

Colunas:
['source_name', 'source_type', 'city_name', 'ingestion_date', 'observation_time', 'temperature_celsius', 'feels_like_celsius', 'relative_humidity_pct', 'precipitation_probability', 'precipitation_mm', 'weather_code', 'wind_speed_10m', 'wind_direction_10m']

Primeiras 5 linhas:


,source_name,source_type,city_name,ingestion_date,observation_time,temperature_celsius,feels_like_celsius,relative_humidity_pct,precipitation_probability,precipitation_mm,weather_code,wind_speed_10m,wind_direction_10m
0,open_meteo,forecast,amsterdam,20260423,2026-04-23T00:00Z,8.8,6.1,80,0,0.0,0,10.8,20
1,open_meteo,forecast,amsterdam,20260423,2026-04-23T01:00Z,8.7,6.1,78,0,0.0,0,9.4,36
2,open_meteo,forecast,amsterdam,20260423,2026-04-23T02:00Z,8.4,5.6,81,0,0.0,0,11.2,33
3,open_meteo,forecast,amsterdam,20260423,2026-04-23T03:00Z,8.0,5.3,82,0,0.0,0,10.4,28
4,open_meteo,forecast,amsterdam,20260423,2026-04-23T04:00Z,8.0,5.4,82,0,0.0,0,9.4,14



Valores ausentes:
source_name                  0
source_type                  0
city_name                    0
ingestion_date               0
observation_time             0
temperature_celsius          0
feels_like_celsius           0
relative_humidity_pct        0
precipitation_probability    0
precipitation_mm             0
weather_code                 0
wind_speed_10m               0
wind_direction_10m           0
dtype: int64


In [36]:
# Verificar os valores ausentes no JSON original
raw_sample = pd.read_json(forecast_files[0])

hourly_sample = raw_sample["hourly"]

print("Valores ausentes no JSON original:")
print(hourly_sample.apply(
    lambda values: pd.Series(values).isna().sum()
))

print("\nValores ausentes no Parquet:")
print(sample_parquet.isna().sum())

Valores ausentes no JSON original:
time                         0
temperature_2m               0
apparent_temperature         0
relative_humidity_2m         0
precipitation_probability    0
precipitation                0
weather_code                 0
wind_speed_10m               0
wind_direction_10m           0
Name: hourly, dtype: int64

Valores ausentes no Parquet:
source_name                  0
source_type                  0
city_name                    0
ingestion_date               0
observation_time             0
temperature_celsius          0
feels_like_celsius           0
relative_humidity_pct        0
precipitation_probability    0
precipitation_mm             0
weather_code                 0
wind_speed_10m               0
wind_direction_10m           0
dtype: int64


In [37]:
# Selecionar o arquivo coletado hoje
source_path = next(
    p for p in forecast_files
    if p.name == "amsterdam_20261015.json"
)

# Transformar JSON em DataFrame
amsterdam_df = build_forecast_dataframe(source_path)

print("Arquivo de origem:", source_path.name)
print("Linhas:", len(amsterdam_df))
print("Colunas:", len(amsterdam_df.columns))
print("\nValores ausentes por coluna:")
print(amsterdam_df.isna().sum())

Arquivo de origem: amsterdam_20261015.json
Linhas: 168
Colunas: 13

Valores ausentes por coluna:
source_name                  0
source_type                  0
city_name                    0
ingestion_date               0
observation_time             0
temperature_celsius          0
feels_like_celsius           0
relative_humidity_pct        0
precipitation_probability    0
precipitation_mm             0
weather_code                 0
wind_speed_10m               0
wind_direction_10m           0
dtype: int64


In [38]:
# Criar o diretório Silver correspondente à data da coleta
output_dir = TARGET_DIR / source_path.parent.name
output_dir.mkdir(parents=True, exist_ok=True)

# Definir o caminho do arquivo Parquet
output_path = output_dir / f"{source_path.stem}.parquet"

# Salvar o DataFrame em Parquet
amsterdam_df.to_parquet(
    output_path,
    engine="pyarrow",
    index=False
)

print("Transformação Bronze → Silver concluída!")
print("Arquivo salvo em:", output_path)
print("Tamanho:", output_path.stat().st_size, "bytes")

Transformação Bronze → Silver concluída!
Arquivo salvo em: /home/mariarita-resende/Projetos/open_meteo_ingestion/data/silver/open_meteo/forecast/20261015/amsterdam_20261015.parquet
Tamanho: 12591 bytes


In [39]:
# Ler novamente o arquivo Parquet
silver_check = pd.read_parquet(output_path, engine="pyarrow")

print("Arquivo lido:", output_path.name)
print("Quantidade de linhas:", len(silver_check))
print("Quantidade de colunas:", len(silver_check.columns))
print("Total de valores ausentes:", silver_check.isna().sum().sum())

print("\nPrimeiras 5 linhas:")
display(silver_check.head())

Arquivo lido: amsterdam_20261015.parquet
Quantidade de linhas: 168
Quantidade de colunas: 13
Total de valores ausentes: 0

Primeiras 5 linhas:


,source_name,source_type,city_name,ingestion_date,observation_time,temperature_celsius,feels_like_celsius,relative_humidity_pct,precipitation_probability,precipitation_mm,weather_code,wind_speed_10m,wind_direction_10m
0,open_meteo,forecast,amsterdam,20261015,2026-10-15T00:00Z,13.4,11.1,78,0,0.0,3,15.1,198
1,open_meteo,forecast,amsterdam,20261015,2026-10-15T01:00Z,13.0,11.1,84,0,0.0,3,13.3,206
2,open_meteo,forecast,amsterdam,20261015,2026-10-15T02:00Z,12.5,10.6,87,0,0.0,3,13.7,188
3,open_meteo,forecast,amsterdam,20261015,2026-10-15T03:00Z,12.7,10.6,84,0,0.0,3,14.0,196
4,open_meteo,forecast,amsterdam,20261015,2026-10-15T04:00Z,12.8,10.8,86,0,0.0,3,14.4,188


In [40]:
# Selecionar somente os arquivos da coleta de 15/10/2026
target_date = "20261015"

new_forecast_files = [
    path
    for path in forecast_files
    if path.parent.name == target_date
]

print("Data da coleta:", target_date)
print("Arquivos selecionados:", len(new_forecast_files))

for path in new_forecast_files:
    print(path.name)

Data da coleta: 20261015
Arquivos selecionados: 14
amsterdam_20261015.json
barcelona_20261015.json
berlin_20261015.json
brussels_20261015.json
budapest_20261015.json
copenhagen_20261015.json
dublin_20261015.json
lisbon_20261015.json
paris_20261015.json
prague_20261015.json
rome_20261015.json
sao_tiago_20261015.json
veghel_20261015.json
vienna_20261015.json


In [41]:
# Processar somente os arquivos da nova coleta
processed_files = 0
processed_rows = 0
processing_errors = []

for path in new_forecast_files:
    try:
        # Transformar JSON em DataFrame
        df = build_forecast_dataframe(path)

        # Criar diretório de destino
        output_dir = TARGET_DIR / target_date
        output_dir.mkdir(parents=True, exist_ok=True)

        # Definir caminho do Parquet
        output_path = output_dir / f"{path.stem}.parquet"

        # Salvar na Silver
        df.to_parquet(
            output_path,
            engine="pyarrow",
            index=False
        )

        processed_files += 1
        processed_rows += len(df)

        print(
            f"OK: {path.name} -> {output_path.name} "
            f"({len(df)} linhas)"
        )

    except Exception as e:
        processing_errors.append((path.name, str(e)))
        print(f"ERRO: {path.name}: {e}")

print("\n========== RESUMO SILVER ==========")
print("Arquivos selecionados:", len(new_forecast_files))
print("Arquivos processados:", processed_files)
print("Linhas processadas:", processed_rows)
print("Arquivos com erro:", len(processing_errors))

OK: amsterdam_20261015.json -> amsterdam_20261015.parquet (168 linhas)
OK: barcelona_20261015.json -> barcelona_20261015.parquet (168 linhas)
OK: berlin_20261015.json -> berlin_20261015.parquet (168 linhas)
OK: brussels_20261015.json -> brussels_20261015.parquet (168 linhas)
OK: budapest_20261015.json -> budapest_20261015.parquet (168 linhas)
OK: copenhagen_20261015.json -> copenhagen_20261015.parquet (168 linhas)
OK: dublin_20261015.json -> dublin_20261015.parquet (168 linhas)
OK: lisbon_20261015.json -> lisbon_20261015.parquet (168 linhas)
OK: paris_20261015.json -> paris_20261015.parquet (168 linhas)
OK: prague_20261015.json -> prague_20261015.parquet (168 linhas)
OK: rome_20261015.json -> rome_20261015.parquet (168 linhas)
OK: sao_tiago_20261015.json -> sao_tiago_20261015.parquet (168 linhas)
OK: veghel_20261015.json -> veghel_20261015.parquet (168 linhas)
OK: vienna_20261015.json -> vienna_20261015.parquet (168 linhas)

========== RESUMO SILVER ==========
Arquivos selecionados: 14

In [42]:
from pathlib import Path
import pandas as pd

# Diretório da coleta de 15/10/2026
silver_dir = TARGET_DIR / "20261015"

silver_files = sorted(silver_dir.glob("*.parquet"))

total_rows = 0
total_nulls = 0
errors = []

print("========== VALIDAÇÃO SILVER ==========")
print("Diretório:", silver_dir)
print("Arquivos encontrados:", len(silver_files))
print()

for path in silver_files:
    try:
        df = pd.read_parquet(path, engine="pyarrow")

        rows = len(df)
        columns = len(df.columns)
        nulls = int(df.isna().sum().sum())

        total_rows += rows
        total_nulls += nulls

        status = (
            "OK"
            if rows == 168 and nulls == 0 and columns == 13
            else "VERIFICAR"
        )

        print(
            f"{path.name}: "
            f"{rows} linhas | "
            f"{columns} colunas | "
            f"{nulls} valores nulos | "
            f"{status}"
        )

    except Exception as e:
        errors.append((path.name, str(e)))
        print(f"ERRO: {path.name}: {e}")

print("\n========== RESUMO ==========")
print("Arquivos encontrados:", len(silver_files))
print("Total de linhas:", total_rows)
print("Total de valores nulos:", total_nulls)
print("Arquivos com erro de leitura:", len(errors))

========== VALIDAÇÃO SILVER ==========
Diretório: /home/mariarita-resende/Projetos/open_meteo_ingestion/data/silver/open_meteo/forecast/20261015
Arquivos encontrados: 14

amsterdam_20261015.parquet: 168 linhas | 13 colunas | 0 valores nulos | OK
barcelona_20261015.parquet: 168 linhas | 13 colunas | 0 valores nulos | OK
berlin_20261015.parquet: 168 linhas | 13 colunas | 0 valores nulos | OK
brussels_20261015.parquet: 168 linhas | 13 colunas | 0 valores nulos | OK
budapest_20261015.parquet: 168 linhas | 13 colunas | 0 valores nulos | OK
copenhagen_20261015.parquet: 168 linhas | 13 colunas | 0 valores nulos | OK
dublin_20261015.parquet: 168 linhas | 13 colunas | 0 valores nulos | OK
lisbon_20261015.parquet: 168 linhas | 13 colunas | 0 valores nulos | OK
paris_20261015.parquet: 168 linhas | 13 colunas | 0 valores nulos | OK
prague_20261015.parquet: 168 linhas | 13 colunas | 0 valores nulos | OK
rome_20261015.parquet: 168 linhas | 13 colunas | 0 valores nulos | OK
sao_tiago_20261015.parquet

In [43]:
# Selecionar somente os arquivos da coleta de 15/10/2026

target_date = "20261015"

source_dir = SOURCE_DIR / target_date
silver_files = sorted(source_dir.glob("*.parquet"))

print("Diretório Silver:", source_dir)
print("Arquivos encontrados:", len(silver_files))

for path in silver_files:
    print(path.name)

Diretório Silver: /home/mariarita-resende/Projetos/open_meteo_ingestion/data/bronze/open_meteo/forecast/20261015
Arquivos encontrados: 0
